In [5]:
from pathlib import Path
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

EPS = 1e-9


# =====================================================
# Parameters
# =====================================================

OUTPUT_DIR = Path("../../tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = Path("../../plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

SOLUTION_DIRS_RELATIVE = {
    "AILS": Path("ails_solutions/subset"),
    "Vanilla": Path("lnsa_solutions/vanilla_subset"),
    "SFC": Path("lnsa_solutions/sfc_subset"),
}

DENSITY_MAP = {
    9: ".90",
    10: ".95",
    11: ".99",
}

COST_LABELS = {
    "linear": "Lin.",
    "convex": "Conv.",
    "concave": "Conc.",
}

METHOD_ORDER = ["AILS", "Vanilla", "SFC"]


# =====================================================
# Locate solution folders
# =====================================================

def find_solution_dirs():
    current = Path.cwd().resolve()

    for parent in [current] + list(current.parents):
        solution_dirs = {
            method: parent / rel_path
            for method, rel_path in SOLUTION_DIRS_RELATIVE.items()
        }

        if all(path.is_dir() for path in solution_dirs.values()):
            print("Found solution folders:")
            for method, path in solution_dirs.items():
                print(f"{method:8s}: {path}")
            return solution_dirs

    raise RuntimeError("Could not find solution folders automatically.")


SOLUTION_DIRS = find_solution_dirs()


# =====================================================
# Parsing helpers
# =====================================================

def read_solution_file(path: Path, method: str) -> dict:
    data = {
        "file": str(path),
        "method": method,
    }

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line == "solution":
                break

            parts = line.split(maxsplit=1)

            if len(parts) != 2:
                continue

            key, value = parts

            if key in {
                "run",
                "seed",
                "iteration_limit",
                "destroy_percent",
                "max_iterations",
                "max_no_improve",
            }:
                data[key] = int(value)
            elif key in {
                "runtime",
                "objective",
                "total_bin_cost",
                "total_excess",
                "total_conflicts",
                "c",
                "alpha",
                "gamma",
                "epsilon",
                "time_limit",
            }:
                data[key] = float(value)
            else:
                data[key] = value

    if "instance_file" not in data:
        raise RuntimeError(f"Missing instance_file in solution file: {path}")

    if "config" not in data:
        raise RuntimeError(f"Missing config in solution file: {path}")

    if "objective" not in data:
        raise RuntimeError(f"Missing objective in solution file: {path}")

    if "runtime" not in data:
        raise RuntimeError(f"Missing runtime in solution file: {path}")

    data["instance_name"] = Path(data["instance_file"]).stem

    return data


def parse_set1_instance(name: str):
    # Correia_Random_x_y_z_t
    m = re.match(r"Correia_Random_(\d+)_(\d+)_(\d+)_(\d+)$", name)

    if not m:
        return None

    x, y, z, t = map(int, m.groups())

    return {
        "set": "S1",
        "x": x,
        "y": y,
        "z": z,
        "t": t,
        "rho": DENSITY_MAP.get(z, str(z)),
    }


def parse_set2_instance(name: str):
    # HS_Random_x_y_z
    m = re.match(r"HS_Random_(\d+)_(\d+)_(\d+)$", name)

    if not m:
        return None

    x, y, z = map(int, m.groups())

    return {
        "set": "S2",
        "x": x,
        "y": y,
        "z": z,
        "rho": DENSITY_MAP.get(y, str(y)),
    }


def parse_config(config: str):
    c = config.lower()

    if "3types" in c:
        bin_label = "3"
    elif "5types" in c:
        bin_label = "5"
    elif "7types" in c:
        bin_label = "7"
    else:
        bin_label = "?"

    if "linear" in c:
        cost_label = COST_LABELS["linear"]
    elif "convex" in c:
        cost_label = COST_LABELS["convex"]
    elif "concave" in c:
        cost_label = COST_LABELS["concave"]
    else:
        cost_label = "?"

    return {
        "bin": bin_label,
        "cost": cost_label,
    }


def add_metadata(row: dict) -> dict:
    meta = parse_set1_instance(row["instance_name"])

    if meta is None:
        meta = parse_set2_instance(row["instance_name"])

    if meta is None:
        row["set"] = "unknown"
        row["rho"] = "?"
    else:
        row.update(meta)

    row.update(parse_config(row["config"]))

    return row


# =====================================================
# Read all solution files
# =====================================================

rows = []

for method, folder in SOLUTION_DIRS.items():
    files = sorted(folder.glob("*.txt"))
    print(f"{method:8s}: {len(files)} solution files")

    for path in files:
        row = read_solution_file(path, method)
        row = add_metadata(row)
        rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError("No solution files found.")

df = df[df["set"].isin(["S1", "S2"])].copy()

if df.empty:
    raise RuntimeError("No Set 1 or Set 2 solution files found.")


# =====================================================
# Compute BKV and gaps
# =====================================================

key_cols = ["instance_name", "config"]

benchmark_df = df[df["method"].isin(["Vanilla", "SFC"])].copy()

if benchmark_df.empty:
    raise RuntimeError("No benchmark results found for Vanilla or SFC.")

bkv = (
    benchmark_df
    .groupby(key_cols, as_index=False)
    .agg(BKV=("objective", "min"))
)

df = df.merge(bkv, on=key_cols, how="left")

if df["BKV"].isna().any():
    missing = df[df["BKV"].isna()][key_cols].drop_duplicates()
    raise RuntimeError(
        "Missing BKV for some instance/config pairs:\n"
        + missing.to_string(index=False)
    )

df["gap"] = 100.0 * (df["objective"] - df["BKV"]) / df["BKV"]

# Numerical cleanup.
df.loc[df["gap"].abs() < EPS, "gap"] = 0.0


# =====================================================
# Overall comparison statistics for text discussion
# =====================================================

overall_stats = (
    df.groupby("method", as_index=False)
    .agg(
        avg_objective=("objective", "mean"),
        avg_gap=("gap", "mean"),
        avg_runtime=("runtime", "mean"),
        runs=("objective", "count"),
    )
)

overall_stats["method"] = pd.Categorical(
    overall_stats["method"],
    categories=METHOD_ORDER,
    ordered=True,
)

overall_stats = overall_stats.sort_values("method")

overall_stats_csv = OUTPUT_DIR / "overall_method_statistics.csv"
overall_stats.to_csv(overall_stats_csv, index=False)

# Extract method-level values.
stats_by_method = overall_stats.set_index("method")

ails_gap = stats_by_method.loc["AILS", "avg_gap"]
vanilla_gap = stats_by_method.loc["Vanilla", "avg_gap"]
sfc_gap = stats_by_method.loc["SFC", "avg_gap"]

ails_runtime = stats_by_method.loc["AILS", "avg_runtime"]
vanilla_runtime = stats_by_method.loc["Vanilla", "avg_runtime"]
sfc_runtime = stats_by_method.loc["SFC", "avg_runtime"]

benchmark_runtime = 0.5 * (vanilla_runtime + sfc_runtime)

runtime_increase_vs_benchmark = (
    100.0 * (ails_runtime - benchmark_runtime) / benchmark_runtime
)

runtime_increase_vs_vanilla = (
    100.0 * (ails_runtime - vanilla_runtime) / vanilla_runtime
)

runtime_increase_vs_sfc = (
    100.0 * (ails_runtime - sfc_runtime) / sfc_runtime
)

# Save a small text file with ready-to-use values.
summary_text_path = OUTPUT_DIR / "overall_method_statistics_summary.txt"

summary_text = f"""Overall method statistics

Average gap:
  AILS    = {ails_gap:.4f}%
  Vanilla = {vanilla_gap:.4f}%
  SFC     = {sfc_gap:.4f}%

Average runtime:
  AILS    = {ails_runtime:.4f} s
  Vanilla = {vanilla_runtime:.4f} s
  SFC     = {sfc_runtime:.4f} s

Runtime comparison:
  AILS vs benchmark average = {runtime_increase_vs_benchmark:.2f}%
  AILS vs Vanilla           = {runtime_increase_vs_vanilla:.2f}%
  AILS vs SFC               = {runtime_increase_vs_sfc:.2f}%
"""

summary_text_path.write_text(summary_text, encoding="utf-8")

# Ready-to-paste LaTeX paragraph.
latex_paragraph_path = OUTPUT_DIR / "overall_method_statistics_paragraph.tex"

latex_paragraph = rf"""As can be observed, AILS consistently obtained negative average gaps,
indicating that it improved the average quality of the solutions and found
new best-known values. It obtained an average gap of ${ails_gap:.2f}\%$,
whereas the average gaps of Vanilla LNSA and SFC-LNSA were
${vanilla_gap:.2f}\%$ and ${sfc_gap:.2f}\%$, respectively. Although AILS
obtained a lower average gap, its average runtime was
${runtime_increase_vs_benchmark:.2f}\%$ higher than the average runtime of
the benchmark methods.
"""

latex_paragraph_path.write_text(latex_paragraph, encoding="utf-8")

print("\nOverall method statistics:")
print(summary_text)

print("Created:")
print(overall_stats_csv)
print(summary_text_path)
print(latex_paragraph_path)


# =====================================================
# Table: Summary statistics by group and method
# =====================================================

group_cols = ["set", "rho", "bin", "cost", "method"]

summary_long = (
    df.groupby(group_cols, as_index=False)
    .agg(
        A=("objective", "mean"),
        G=("gap", "mean"),
        T=("runtime", "mean"),
    )
)

summary_long["method"] = pd.Categorical(
    summary_long["method"],
    categories=METHOD_ORDER,
    ordered=True,
)

summary_long = summary_long.sort_values(["set", "rho", "bin", "cost", "method"])

# Save long CSV.
summary_csv = OUTPUT_DIR / "method_statistics_by_group.csv"
summary_long.to_csv(summary_csv, index=False)


def group_sort_key(row):
    set_rank = 1 if row["set"] == "S1" else 2
    rho_rank = {".90": 90, ".95": 95, ".99": 99}.get(row["rho"], 999)
    bin_rank = int(row["bin"]) if str(row["bin"]).isdigit() else 999
    cost_rank = {"Lin.": 1, "Conv.": 2, "Conc.": 3}.get(row["cost"], 999)
    return (set_rank, rho_rank, bin_rank, cost_rank)


def build_latex_summary_table(summary: pd.DataFrame) -> str:
    rows_out = []

    table_index = {}
    for _, r in summary.iterrows():
        key = (r["set"], r["rho"], r["bin"], r["cost"])
        method = r["method"]

        if key not in table_index:
            table_index[key] = {}

        table_index[key][method] = {
            "A": r["A"],
            "G": r["G"],
            "T": r["T"],
        }

    ordered_keys = sorted(
        table_index.keys(),
        key=lambda k: group_sort_key({
            "set": k[0],
            "rho": k[1],
            "bin": k[2],
            "cost": k[3],
        })
    )

    for key in ordered_keys:
        set_name, rho, bin_label, cost = key
        cells = [set_name, rho, bin_label, cost]

        for method in METHOD_ORDER:
            vals = table_index[key].get(method)

            if vals is None:
                cells.extend(["--", "--", "--"])
            else:
                cells.extend([
                    f"{vals['A']:.1f}",
                    f"{vals['G']:.2f}",
                    f"{vals['T']:.1f}",
                ])

        rows_out.append(" & ".join(cells) + r" \\")

    body = "\n".join(rows_out)

    return rf"""\begin{{table}}[!ht]
\centering
\footnotesize
\caption{{Summary statistics by instance group and method.}}
\label{{tab:method_statistics_by_group}}
\begin{{tabular}}{{cccc|ccc|ccc|ccc}}
\hline
\multirow{{2}}{{*}}{{\textbf{{Set}}}} &
\multirow{{2}}{{*}}{{\textbf{{$\rho$}}}} &
\multirow{{2}}{{*}}{{\textbf{{Bin}}}} &
\multirow{{2}}{{*}}{{\textbf{{Cost}}}} &
\multicolumn{{3}}{{c|}}{{\textbf{{AILS}}}} &
\multicolumn{{3}}{{c|}}{{\textbf{{Vanilla}}}} &
\multicolumn{{3}}{{c}}{{\textbf{{SFC}}}} \\
\cline{{5-13}}
& & & &
\textbf{{A}} & \textbf{{G}} & \textbf{{T}} &
\textbf{{A}} & \textbf{{G}} & \textbf{{T}} &
\textbf{{A}} & \textbf{{G}} & \textbf{{T}} \\
\hline
{body}
\hline
\end{{tabular}}
\end{{table}}
"""


summary_tex = OUTPUT_DIR / "table_method_statistics_by_group.tex"
summary_tex.write_text(build_latex_summary_table(summary_long), encoding="utf-8")


# =====================================================
# Wilcoxon paired tests
# =====================================================

def paired_wilcoxon(df: pd.DataFrame, other_method: str, metric: str):
    """
    One-sided Wilcoxon signed-rank test.

    H1:
        AILS metric < other_method metric

    Pairing:
        instance_name, config, run if available;
        otherwise instance_name, config, seed.
    """
    if "run" in df.columns and df["run"].notna().any():
        pair_cols = ["instance_name", "config", "run"]
    elif "seed" in df.columns and df["seed"].notna().any():
        pair_cols = ["instance_name", "config", "seed"]
    else:
        raise RuntimeError(
            "Cannot perform paired test: neither run nor seed is available."
        )

    sub = df[df["method"].isin(["AILS", other_method])].copy()

    wide = sub.pivot_table(
        index=pair_cols,
        columns="method",
        values=metric,
        aggfunc="mean",
    ).dropna(subset=["AILS", other_method])

    if wide.empty:
        return {
            "comparison": f"AILS_vs_{other_method}",
            "metric": metric,
            "n_pairs": 0,
            "ails_mean": np.nan,
            "other_mean": np.nan,
            "mean_difference_ails_minus_other": np.nan,
            "p_value": np.nan,
            "alternative": "AILS < other",
        }

    diff = wide["AILS"] - wide[other_method]

    # Wilcoxon fails if all differences are zero.
    if np.allclose(diff.to_numpy(), 0.0):
        p_value = 1.0
    else:
        result = wilcoxon(
            wide["AILS"],
            wide[other_method],
            alternative="less",
            zero_method="wilcox",
        )
        p_value = result.pvalue

    return {
        "comparison": f"AILS_vs_{other_method}",
        "metric": metric,
        "n_pairs": len(wide),
        "ails_mean": wide["AILS"].mean(),
        "other_mean": wide[other_method].mean(),
        "mean_difference_ails_minus_other": diff.mean(),
        "p_value": p_value,
        "alternative": "AILS < other",
    }


wilcoxon_rows = []

for other in ["Vanilla", "SFC"]:
    for metric in ["objective", "gap"]:
        wilcoxon_rows.append(paired_wilcoxon(df, other, metric))

wilcoxon_df = pd.DataFrame(wilcoxon_rows)

wilcoxon_csv = OUTPUT_DIR / "wilcoxon_tests.csv"
wilcoxon_df.to_csv(wilcoxon_csv, index=False)


def build_latex_wilcoxon_table(wdf: pd.DataFrame) -> str:
    rows = []

    metric_label = {
        "objective": "Objective",
        "gap": "Gap",
    }

    for _, r in wdf.iterrows():
        comparison = r["comparison"].replace("AILS_vs_", "AILS vs. ")
        metric = metric_label.get(r["metric"], r["metric"])

        rows.append(
            f"{comparison} & "
            f"{metric} & "
            f"{int(r['n_pairs'])} & "
            f"{r['ails_mean']:.4f} & "
            f"{r['other_mean']:.4f} & "
            f"{r['mean_difference_ails_minus_other']:.4f} & "
            f"{r['p_value']:.4g} \\\\"
        )

    body = "\n".join(rows)

    return rf"""\begin{{table}}[!ht]
\centering
\footnotesize
\caption{{Paired Wilcoxon signed-rank tests comparing AILS against LNSA variants.}}
\label{{tab:wilcoxon_ails}}
\begin{{tabular}}{{llrrrrr}}
\hline
\textbf{{Comparison}} &
\textbf{{Metric}} &
\textbf{{Pairs}} &
\textbf{{AILS mean}} &
\textbf{{Other mean}} &
\textbf{{$\Delta$}} &
\textbf{{$p$-value}} \\
\hline
{body}
\hline
\end{{tabular}}
\end{{table}}
"""


wilcoxon_tex = OUTPUT_DIR / "table_wilcoxon_tests.tex"
wilcoxon_tex.write_text(build_latex_wilcoxon_table(wilcoxon_df),encoding="utf-8")


# =====================================================
# Empirical CDF of average gap
# =====================================================

# Remove only numerical noise at the run level.
df["gap"] = pd.to_numeric(df["gap"], errors="coerce")
df.loc[df["gap"].abs() < EPS, "gap"] = 0.0

# Average gap per instance/config/method.
avg_gap = (
    df.groupby(["method", "instance_name", "config"], as_index=False)
    .agg(avg_gap=("gap", "mean"))
)

# Remove only numerical noise after averaging.
avg_gap.loc[avg_gap["avg_gap"].abs() < EPS, "avg_gap"] = 0.0

# Save ECDF input data.
ecdf_csv = OUTPUT_DIR / "ecdf_average_gap.csv"
avg_gap.to_csv(ecdf_csv, index=False)

# Optional sanity check: AILS cases worse than the Vanilla/SFC BKV.
ails_positive = (
    avg_gap[
        (avg_gap["method"] == "AILS") &
        (avg_gap["avg_gap"] > EPS)
    ]
    .sort_values("avg_gap", ascending=False)
)

if not ails_positive.empty:
    print("\nAILS instance/config pairs with positive average gap:")
    print(
        ails_positive[
            ["instance_name", "config", "avg_gap"]
        ].to_string(index=False)
    )

# Plot ECDF.
plt.figure(figsize=(7.0, 4.5))

for method in METHOD_ORDER:
    values = (
        avg_gap
        .loc[avg_gap["method"] == method, "avg_gap"]
        .dropna()
        .to_numpy()
    )

    if len(values) == 0:
        continue

    values = np.sort(values)
    y = np.arange(1, len(values) + 1) / len(values)

    plt.step(values, y, where="post", label=method)

plt.axvline(0.0, linestyle="--", linewidth=1)
plt.xlabel("Average gap to Vanilla/SFC BKV (%)")
plt.ylabel("Empirical CDF")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

ecdf_plot = PLOT_DIR / "ecdf_average_gap.png"
plt.savefig(ecdf_plot, dpi=300)
plt.close()


# =====================================================
# Final messages
# =====================================================

print("Created:")
print(summary_csv)
print(wilcoxon_csv)
print(ecdf_csv)
print(summary_tex)
print(wilcoxon_tex)
print(ecdf_plot)

Found solution folders:
AILS    : /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/ails_solutions/subset
Vanilla : /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/vanilla_subset
SFC     : /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/sfc_subset
AILS    : 1500 solution files
Vanilla : 1500 solution files
SFC     : 1500 solution files

Overall method statistics:
Overall method statistics

Average gap:
  AILS    = -0.2682%
  Vanilla = 0.0630%
  SFC     = 0.1991%

Average runtime:
  AILS    = 50.6899 s
  Vanilla = 11.2412 s
  SFC     = 11.4302 s

Runtime comparison:
  AILS vs benchmark average = 347.17%
  AILS vs Vanilla           = 350.93%
  AILS vs SFC               = 343.47%

Created:
../../tables/ov